In [3]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "data_2000_2019.csv"

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.info()

(133920, 26)
<class 'pandas.DataFrame'>
RangeIndex: 133920 entries, 0 to 133919
Data columns (total 26 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   micro_code       133920 non-null  int64  
 1   micro_name       133920 non-null  str    
 2   micro_name_ibge  133920 non-null  str    
 3   meso_code        133920 non-null  int64  
 4   meso_name        133920 non-null  str    
 5   state_code       133920 non-null  int64  
 6   state_name       133920 non-null  str    
 7   region_code      133920 non-null  int64  
 8   region_name      133920 non-null  str    
 9   biome_code       133920 non-null  int64  
 10  biome_name       133920 non-null  str    
 11  ecozone_code     133920 non-null  int64  
 12  ecozone_name     133920 non-null  str    
 13  main_climate     133920 non-null  str    
 14  month            133920 non-null  int64  
 15  year             133920 non-null  int64  
 16  time             133920 non-null  in

In [4]:
print(df[["year", "month", "time"]].head(20))
print(df[["year", "month", "time"]].tail(20))

print("\nUnique years:", df["year"].nunique())
print("Year range:", df["year"].min(), "→", df["year"].max())
print("Unique months:", sorted(df["month"].unique()))
print("Unique microregions:", df["micro_code"].nunique())

    year  month  time
0   2000      1     1
1   2000      1     1
2   2000      1     1
3   2000      1     1
4   2000      1     1
5   2000      1     1
6   2000      1     1
7   2000      1     1
8   2000      1     1
9   2000      1     1
10  2000      1     1
11  2000      1     1
12  2000      1     1
13  2000      1     1
14  2000      1     1
15  2000      1     1
16  2000      1     1
17  2000      1     1
18  2000      1     1
19  2000      1     1
        year  month  time
133900  2019     12   240
133901  2019     12   240
133902  2019     12   240
133903  2019     12   240
133904  2019     12   240
133905  2019     12   240
133906  2019     12   240
133907  2019     12   240
133908  2019     12   240
133909  2019     12   240
133910  2019     12   240
133911  2019     12   240
133912  2019     12   240
133913  2019     12   240
133914  2019     12   240
133915  2019     12   240
133916  2019     12   240
133917  2019     12   240
133918  2019     12   240
133919  2019     1

In [5]:
# ============================================================
# STEP 1 — PANEL UNIQUENESS CHECK
# ============================================================

key = ["micro_code", "year", "month"]

duplicate_rows = df.duplicated(subset=key, keep=False)

print("Duplicate microregion-month observations:", duplicate_rows.sum())

if duplicate_rows.sum() > 0:
    print("\nDuplicate observations:")
    display(
        df.loc[
            duplicate_rows,
            key + ["micro_name", "dengue_cases"]
        ].sort_values(key).head(50)
    )
else:
    print("✓ Each microregion has at most one observation per month.")

Duplicate microregion-month observations: 0
✓ Each microregion has at most one observation per month.


In [6]:
# ============================================================
# STEP 2 — BALANCED PANEL CHECK
# ============================================================

months_per_micro = (
    df.groupby("micro_code")["time"]
      .nunique()
)

print("Number of microregions:", months_per_micro.size)
print("Minimum periods per microregion:", months_per_micro.min())
print("Maximum periods per microregion:", months_per_micro.max())
print("Expected periods per microregion:", 240)

# Identify any incomplete microregions
incomplete = months_per_micro[months_per_micro != 240]

if incomplete.empty:
    print("\n✓ Balanced panel confirmed.")
    print("✓ Every microregion contains all 240 monthly periods.")
else:
    print("\n⚠ Incomplete microregions found:")
    print(incomplete)

Number of microregions: 558
Minimum periods per microregion: 240
Maximum periods per microregion: 240
Expected periods per microregion: 240

✓ Balanced panel confirmed.
✓ Every microregion contains all 240 monthly periods.


In [7]:
# ============================================================
# STEP 3 — JOINT MISSINGNESS STRUCTURE
# ============================================================

missing_cols = [
    "dengue_cases",
    "population",
    "pop_density",
    "pdsi"
]

missing_pattern = (
    df[missing_cols]
    .isna()
    .value_counts(dropna=False)
    .rename("n")
    .reset_index()
)

missing_pattern["percentage"] = (
    100 * missing_pattern["n"] / len(df)
)

missing_pattern

,dengue_cases,population,pop_density,pdsi,n,percentage
0,False,False,False,False,126996,94.829749
1,True,True,True,False,6684,4.991039
2,False,False,False,True,228,0.170251
3,True,True,True,True,12,0.008961


In [9]:
# ============================================================
# RESTORE / VERIFY DATE COLUMN
# ============================================================

if "date" not in df.columns:
    df["date"] = pd.to_datetime(
        dict(
            year=df["year"],
            month=df["month"],
            day=1
        )
    )

df = df.sort_values(["micro_code", "date"]).reset_index(drop=True)

print("Date column restored.")
print("Date range:", df["date"].min(), "to", df["date"].max())
print("Number of unique dates:", df["date"].nunique())

Date column restored.
Date range: 2000-01-01 00:00:00 to 2019-12-01 00:00:00
Number of unique dates: 240


In [10]:
# ============================================================
# STEP 4 — TEMPORAL STRUCTURE OF MISSINGNESS
# ============================================================

target_missing = df["dengue_cases"].isna()

missing_by_time = (
    df.loc[target_missing]
      .groupby(["year", "month", "date"])
      .size()
      .reset_index(name="n_missing")
      .sort_values("date")
)

print("Total missing dengue observations:", target_missing.sum())
print("\nNumber of months affected:", missing_by_time["date"].nunique())
print("Total months in dataset:", df["date"].nunique())

print("\nFirst 20 affected months:")
display(missing_by_time.head(20))

print("\nLast 20 affected months:")
display(missing_by_time.tail(20))

Total missing dengue observations: 6696

Number of months affected: 12
Total months in dataset: 240

First 20 affected months:


,year,month,date,n_missing
0,2000,1,2000-01-01,558
1,2000,2,2000-02-01,558
2,2000,3,2000-03-01,558
3,2000,4,2000-04-01,558
4,2000,5,2000-05-01,558
5,2000,6,2000-06-01,558
6,2000,7,2000-07-01,558
7,2000,8,2000-08-01,558
8,2000,9,2000-09-01,558
9,2000,10,2000-10-01,558



Last 20 affected months:


,year,month,date,n_missing
0,2000,1,2000-01-01,558
1,2000,2,2000-02-01,558
2,2000,3,2000-03-01,558
3,2000,4,2000-04-01,558
4,2000,5,2000-05-01,558
5,2000,6,2000-06-01,558
6,2000,7,2000-07-01,558
7,2000,8,2000-08-01,558
8,2000,9,2000-09-01,558
9,2000,10,2000-10-01,558


In [11]:
# ============================================================
# STEP 4B — EXACT MONTHS WITH MISSING DENGUE OBSERVATIONS
# ============================================================

missing_by_time = (
    df.loc[df["dengue_cases"].isna()]
      .groupby("date")
      .size()
      .reset_index(name="n_missing")
      .sort_values("date")
)

missing_by_time["percentage_of_regions"] = (
    100 * missing_by_time["n_missing"] / df["micro_code"].nunique()
)

missing_by_time

,date,n_missing,percentage_of_regions
0,2000-01-01,558,100.0
1,2000-02-01,558,100.0
2,2000-03-01,558,100.0
3,2000-04-01,558,100.0
4,2000-05-01,558,100.0
5,2000-06-01,558,100.0
6,2000-07-01,558,100.0
7,2000-08-01,558,100.0
8,2000-09-01,558,100.0
9,2000-10-01,558,100.0


In [12]:
# ============================================================
# STEP 5 — 2000 DATA AVAILABILITY BY VARIABLE
# ============================================================

variables_to_check = [
    "dengue_cases",
    "population",
    "pop_density",
    "tmax",
    "tmin",
    "pdsi",
    "urban",
    "water_network",
    "water_shortage"
]

availability_2000 = pd.DataFrame({
    "missing_count": df.loc[df["year"] == 2000, variables_to_check].isna().sum(),
    "non_missing_count": df.loc[df["year"] == 2000, variables_to_check].notna().sum(),
})

availability_2000["missing_percentage"] = (
    100 * availability_2000["missing_count"] / (558 * 12)
)

availability_2000

,missing_count,non_missing_count,missing_percentage
dengue_cases,6696,0,100.000000
population,6696,0,100.000000
pop_density,6696,0,100.000000
tmax,0,6696,0.000000
tmin,0,6696,0.000000
pdsi,12,6684,0.179211
urban,0,6696,0.000000
water_network,0,6696,0.000000
water_shortage,0,6696,0.000000


In [13]:
# ============================================================
# STEP 6 — LOCATE MISSING PDSI OBSERVATIONS
# ============================================================

pdsi_missing = df.loc[
    df["pdsi"].isna(),
    [
        "micro_code",
        "micro_name",
        "state_name",
        "year",
        "month",
        "date",
        "pdsi"
    ]
].sort_values(["date", "micro_code"])

print("Total missing PDSI observations:", len(pdsi_missing))

display(pdsi_missing)

Total missing PDSI observations: 240


,micro_code,micro_name,state_name,year,month,date,pdsi
46320,26019,Fern.Noronha,Pernambuco,2000,1,2000-01-01,NaN
46321,26019,Fern.Noronha,Pernambuco,2000,2,2000-02-01,NaN
46322,26019,Fern.Noronha,Pernambuco,2000,3,2000-03-01,NaN
46323,26019,Fern.Noronha,Pernambuco,2000,4,2000-04-01,NaN
46324,26019,Fern.Noronha,Pernambuco,2000,5,2000-05-01,NaN
...,...,...,...,...,...,...,...
46555,26019,Fern.Noronha,Pernambuco,2019,8,2019-08-01,NaN
46556,26019,Fern.Noronha,Pernambuco,2019,9,2019-09-01,NaN
46557,26019,Fern.Noronha,Pernambuco,2019,10,2019-10-01,NaN
46558,26019,Fern.Noronha,Pernambuco,2019,11,2019-11-01,NaN


In [14]:
# ============================================================
# STEP 7 — FORMAL PDSI MISSINGNESS STRUCTURE
# ============================================================

pdsi_missing_summary = (
    df.assign(pdsi_missing=df["pdsi"].isna())
      .groupby(["micro_code", "micro_name", "state_name"])["pdsi_missing"]
      .agg(
          missing_count="sum",
          total_count="size"
      )
      .reset_index()
)

pdsi_missing_summary["missing_percentage"] = (
    100
    * pdsi_missing_summary["missing_count"]
    / pdsi_missing_summary["total_count"]
)

pdsi_missing_summary = (
    pdsi_missing_summary[
        pdsi_missing_summary["missing_count"] > 0
    ]
    .sort_values("missing_count", ascending=False)
)

pdsi_missing_summary

,micro_code,micro_name,state_name,missing_count,total_count,missing_percentage
193,26019,Fern.Noronha,Pernambuco,240,240,100.0


In [15]:
# ============================================================
# STEP 8 — FERNANDO DE NORONHA PROFILE
# ============================================================

noronha = df[df["micro_code"] == 26019].copy()

profile_variables = [
    "dengue_cases",
    "population",
    "pop_density",
    "tmax",
    "tmin",
    "pdsi",
    "urban",
    "water_network",
    "water_shortage"
]

noronha_profile = pd.DataFrame({
    "missing_count": noronha[profile_variables].isna().sum(),
    "non_missing_count": noronha[profile_variables].notna().sum(),
    "unique_values": noronha[profile_variables].nunique(dropna=True)
})

noronha_profile

,missing_count,non_missing_count,unique_values
dengue_cases,12,228,30
population,12,228,19
pop_density,12,228,19
tmax,0,240,35
tmin,0,240,28
pdsi,240,0,0
urban,0,240,1
water_network,0,240,1
water_shortage,0,240,1


In [16]:
# ============================================================
# STEP 9 — PDSI RELATIONSHIP WITH OBSERVED CLIMATE VARIABLES
# ============================================================

climate_vars = [
    "pdsi",
    "tmax",
    "tmin",
    "urban",
    "water_network",
    "water_shortage"
]

pdsi_corr = (
    df[climate_vars]
    .corr(method="spearman")["pdsi"]
    .sort_values(ascending=False)
)

pdsi_corr

pdsi              1.000000
urban             0.093167
water_network     0.016886
water_shortage   -0.007049
tmin             -0.100181
tmax             -0.139091
Name: pdsi, dtype: float64

In [17]:
# ============================================================
# STEP 10 — PDSI SEASONAL STRUCTURE
# ============================================================

pdsi_monthly = (
    df.dropna(subset=["pdsi"])
      .groupby("month")["pdsi"]
      .agg(
          mean="mean",
          median="median",
          std="std",
          min="min",
          max="max"
      )
      .reset_index()
)

pdsi_monthly

,month,mean,median,std,min,max
0,1,-0.346899,-0.408287,1.992805,-5.940512,6.653169
1,2,-0.383202,-0.492301,1.925994,-5.907659,7.479940
2,3,-0.414869,-0.525598,1.925615,-6.203355,6.849275
3,4,-0.414076,-0.573978,1.944619,-6.210854,6.564851
4,5,-0.416044,-0.588799,1.919185,-5.842462,7.009936
5,6,-0.425705,-0.536351,1.895656,-5.644984,6.665838
6,7,-0.431849,-0.533027,1.864695,-5.786014,5.922831
7,8,-0.518486,-0.605661,1.809956,-5.805633,6.356507
8,9,-0.569751,-0.677610,1.806372,-5.463077,5.521915
9,10,-0.586829,-0.705363,1.868745,-6.087390,5.940010


In [18]:
pdsi_region_stats = (
    df.dropna(subset=["pdsi"])
      .groupby(["micro_code", "micro_name"])["pdsi"]
      .agg(
          mean="mean",
          std="std",
          min="min",
          max="max",
          n="count"
      )
      .reset_index()
)

pdsi_region_stats.describe()

,micro_code,mean,std,min,max,n
count,557.000000,557.000000,557.000000,557.000000,557.000000,557.0
mean,31329.102334,-0.459147,1.667009,-3.939463,3.776401,240.0
std,10403.572425,0.805694,0.472567,0.997239,1.567602,0.0
min,11001.000000,-2.697026,0.324528,-7.634145,0.270883,240.0
25%,24007.000000,-0.951128,1.314552,-4.638983,2.416768,240.0
50%,31028.000000,-0.522806,1.703036,-3.984386,3.793031,240.0
75%,41007.000000,0.088797,1.976655,-3.354835,5.074831,240.0
max,53001.000000,2.111614,2.782478,-0.885887,7.479940,240.0


In [19]:
comparison_vars = [
    "tmax",
    "tmin",
    "urban",
    "water_network",
    "water_shortage"
]

noronha_mean = (
    df[df["micro_code"] == 26019][comparison_vars]
    .mean()
)

region_means = (
    df.groupby(["micro_code", "micro_name"])[comparison_vars]
      .mean()
)

distance = (
    (region_means - noronha_mean)
    .pow(2)
    .sum(axis=1)
    .pow(0.5)
    .sort_values()
)

nearest_regions = (
    region_means.loc[distance.head(10).index]
    .copy()
)

nearest_regions["distance"] = distance.head(10)

nearest_regions

,,tmax,tmin,urban,water_network,water_shortage,distance
micro_code,micro_name,,,,,,
26019,Fern.Noronha,29.689584,23.714584,100.00,95.74,0.1746,5.024296e-15
24018,Natal,30.450834,22.300834,99.14,97.63,0.0000,2.630661e+00
28011,Aracaju,29.711401,22.195316,97.46,96.44,0.0349,3.044629e+00
29021,Salvador,29.614078,21.634702,98.75,97.79,0.5151,3.195708e+00
32009,Vit<f3>ria,28.914793,20.863816,98.81,97.59,0.5413,3.701385e+00
25022,Jo<e3>o Pessoa,29.865143,22.270416,97.20,93.71,0.1164,3.752430e+00
23016,Fortaleza,31.068188,23.434082,97.49,89.94,0.8342,6.508026e+00
35005,Catanduva,30.154587,18.231286,96.55,94.98,0.7991,6.569085e+00
35017,Ara<e7>atuba,30.776990,18.662668,95.76,95.39,0.5488,6.704060e+00


In [20]:
pdsi_dengue = (
    df[["pdsi", "dengue_cases"]]
    .dropna()
    .corr(method="spearman")
)

pdsi_dengue

,pdsi,dengue_cases
pdsi,1.000000,-0.067787
dengue_cases,-0.067787,1.000000


In [21]:
df.groupby("year").agg(
    total_rows=("micro_code", "size"),
    dengue_observed=("dengue_cases", "count"),
    population_observed=("population", "count"),
    pdsi_observed=("pdsi", "count")
).head(5)

,total_rows,dengue_observed,population_observed,pdsi_observed
year,,,,
2000,6696,0,0,6684
2001,6696,6696,6696,6684
2002,6696,6696,6696,6684
2003,6696,6696,6696,6684
2004,6696,6696,6696,6684


In [22]:
population_2001 = (
    df[df["year"] == 2001]
    .groupby("micro_code")["population"]
    .agg(
        min="min",
        max="max",
        mean="mean",
        n="count"
    )
)

population_2001["range"] = (
    population_2001["max"] - population_2001["min"]
)

population_2001.describe()

,min,max,mean,n,range
count,5.580000e+02,5.580000e+02,5.580000e+02,558.0,558.0
mean,3.089352e+05,3.089352e+05,3.089352e+05,12.0,0.0
std,8.094048e+05,8.094048e+05,8.094048e+05,0.0,0.0
min,2.096000e+03,2.096000e+03,2.096000e+03,12.0,0.0
25%,9.185975e+04,9.185975e+04,9.185975e+04,12.0,0.0
50%,1.578225e+05,1.578225e+05,1.578225e+05,12.0,0.0
75%,2.672605e+05,2.672605e+05,2.672605e+05,12.0,0.0
max,1.288620e+07,1.288620e+07,1.288620e+07,12.0,0.0


In [23]:
population_yearly = (
    df.groupby(["micro_code", "year"])["population"]
      .first()
      .reset_index()
)

population_changes = (
    population_yearly
    .sort_values(["micro_code", "year"])
    .assign(
        population_change=lambda x: (
            x.groupby("micro_code")["population"].pct_change() * 100
        )
    )
)

population_changes["population_change"].describe()

count    10044.000000
mean         1.009359
std          2.223504
min        -32.842155
25%          0.301989
50%          0.801659
75%          1.702428
max         30.554502
Name: population_change, dtype: float64

In [24]:
extreme_population_changes = (
    population_changes[
        population_changes["population_change"].abs() > 10
    ]
    .sort_values("population_change")
)

extreme_population_changes

,micro_code,year,population,population_change
287,13002,2007,22788.0,-32.842155
870,15014,2010,209209.0,-20.846213
87,11005,2007,68034.0,-19.931741
10707,51019,2007,70653.0,-17.454698
367,13006,2007,127589.0,-15.865584
...,...,...,...,...
3867,26019,2007,2801.0,20.680741
6787,33009,2007,123198.0,22.682733
950,15018,2010,166420.0,26.015614
487,13012,2007,67825.0,28.206340


In [25]:
extreme_codes = extreme_population_changes["micro_code"].unique()

population_trajectories = (
    population_yearly[
        population_yearly["micro_code"].isin(extreme_codes)
    ]
    .sort_values(["micro_code", "year"])
)

population_trajectories

,micro_code,year,population
0,11001,2000,NaN
1,11001,2001,426640.0
2,11001,2002,437202.0
3,11001,2003,448419.0
4,11001,2004,497775.0
...,...,...,...
11135,52018,2015,121515.0
11136,52018,2016,122993.0
11137,52018,2017,124433.0
11138,52018,2018,126788.0


In [26]:
extreme_pairs = extreme_population_changes[
    ["micro_code", "year", "population_change"]
].copy()

rows = []

for _, r in extreme_pairs.iterrows():
    code = r["micro_code"]
    year = int(r["year"])

    temp = population_yearly[
        (population_yearly["micro_code"] == code) &
        (population_yearly["year"].between(year - 2, year + 2))
    ].copy()

    temp["trigger_year"] = year
    temp["trigger_change"] = r["population_change"]

    rows.append(temp)

extreme_windows = (
    pd.concat(rows)
      .sort_values(["micro_code", "year"])
      .reset_index(drop=True)
)

extreme_windows

,micro_code,year,population,trigger_year,trigger_change
0,11001,2002,437202.0,2004,11.006670
1,11001,2003,448419.0,2004,11.006670
2,11001,2004,497775.0,2004,11.006670
3,11001,2005,485002.0,2004,11.006670
4,11001,2006,497936.0,2004,11.006670
...,...,...,...,...,...
353,52018,2008,98317.0,2010,12.270697
354,52018,2009,98723.0,2010,12.270697
355,52018,2010,110837.0,2010,12.270697
356,52018,2011,112141.0,2010,12.270697


In [27]:
density_corr = (
    df[["population", "pop_density"]]
    .dropna()
    .corr(method="spearman")
)

density_corr

,population,pop_density
population,1.000000,0.483475
pop_density,0.483475,1.000000


In [28]:
density_consistency = (
    df.groupby(["micro_code", "year"])
      .agg(
          population_min=("population", "min"),
          population_max=("population", "max"),
          density_min=("pop_density", "min"),
          density_max=("pop_density", "max")
      )
      .reset_index()
)

density_consistency["population_range"] = (
    density_consistency["population_max"]
    - density_consistency["population_min"]
)

density_consistency["density_range"] = (
    density_consistency["density_max"]
    - density_consistency["density_min"]
)

density_consistency[
    ["population_range", "density_range"]
].describe()

,population_range,density_range
count,10602.0,10602.0
mean,0.0,0.0
std,0.0,0.0
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,0.0,0.0
max,0.0,0.0


In [29]:
density_yearly = (
    df.groupby(["micro_code", "year"])["pop_density"]
      .first()
      .reset_index()
)

density_changes = (
    density_yearly
    .sort_values(["micro_code", "year"])
    .assign(
        density_change=lambda x: (
            x.groupby("micro_code")["pop_density"].pct_change() * 100
        )
    )
)

density_changes["density_change"].describe()

count    10044.000000
mean         1.009359
std          2.223504
min        -32.842155
25%          0.301989
50%          0.801659
75%          1.702428
max         30.554502
Name: density_change, dtype: float64

In [30]:
df["implied_area"] = (
    df["population"] / df["pop_density"]
)

area_check = (
    df.groupby("micro_code")["implied_area"]
      .agg(
          mean="mean",
          std="std",
          min="min",
          max="max"
      )
      .reset_index()
)

area_check["relative_std"] = (
    area_check["std"] / area_check["mean"]
)

area_check.describe()

,micro_code,mean,std,min,max,relative_std
count,558.000000,558.000000,5.580000e+02,558.000000,558.000000,5.580000e+02
mean,31319.586022,15237.583766,2.139370e-11,15237.583766,15237.583766,1.263910e-15
std,10396.659812,29513.160746,5.241545e-11,29513.160746,29513.160746,6.644049e-16
min,11001.000000,12.994238,1.789416e-14,12.994238,12.994238,2.952483e-16
25%,24007.250000,2861.835883,2.876787e-12,2861.835883,2861.835883,7.092801e-16
50%,31027.500000,5548.540199,6.234727e-12,5548.540199,5548.540199,1.108050e-15
75%,41006.750000,15958.869794,1.884160e-11,15958.869794,15958.869794,1.780307e-15
max,53001.000000,332294.738547,6.553211e-10,332294.738547,332294.738547,3.855459e-15


In [31]:
df.drop(columns=["implied_area"], inplace=True)

In [32]:
dengue_validity = {
    "negative_count": (df["dengue_cases"] < 0).sum(),
    "zero_count": (df["dengue_cases"] == 0).sum(),
    "positive_count": (df["dengue_cases"] > 0).sum(),
    "missing_count": df["dengue_cases"].isna().sum(),
}

dengue_validity

{'negative_count': np.int64(0),
 'zero_count': np.int64(34031),
 'positive_count': np.int64(93193),
 'missing_count': np.int64(6696)}

In [33]:
population_validity = {
    "negative_count": (df["population"] < 0).sum(),
    "zero_count": (df["population"] == 0).sum(),
    "positive_count": (df["population"] > 0).sum(),
    "missing_count": df["population"].isna().sum(),
}

population_validity

{'negative_count': np.int64(0),
 'zero_count': np.int64(0),
 'positive_count': np.int64(127224),
 'missing_count': np.int64(6696)}

In [34]:
density_validity = {
    "negative_count": (df["pop_density"] < 0).sum(),
    "zero_count": (df["pop_density"] == 0).sum(),
    "positive_count": (df["pop_density"] > 0).sum(),
    "missing_count": df["pop_density"].isna().sum(),
}

density_validity

{'negative_count': np.int64(0),
 'zero_count': np.int64(0),
 'positive_count': np.int64(127224),
 'missing_count': np.int64(6696)}

In [35]:
temperature_validity = df[["tmax", "tmin"]].agg(
    [
        "count",
        "min",
        "max"
    ]
)

temperature_validity

,tmax,tmin
count,133920.000000,133920.000000
min,14.425291,4.615391
max,37.885544,25.765200


In [36]:
temp_inconsistency = df["tmin"] > df["tmax"]

{
    "tmin_greater_than_tmax": temp_inconsistency.sum(),
    "percentage": 100 * temp_inconsistency.mean()
}

{'tmin_greater_than_tmax': np.int64(0), 'percentage': np.float64(0.0)}

In [37]:
temperature_corr = (
    df[["tmax", "tmin"]]
    .corr(method="pearson")
)

temperature_corr

,tmax,tmin
tmax,1.000000,0.858772
tmin,0.858772,1.000000


In [38]:
pdsi_validity = df["pdsi"].agg(
    [
        "count",
        "min",
        "max",
        "mean",
        "std"
    ]
)

pdsi_validity

count    133680.000000
min          -7.634145
max           7.479940
mean         -0.459147
std           1.907180
Name: pdsi, dtype: float64

In [39]:
pdsi_quantiles = df["pdsi"].quantile(
    [0.001, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
)

pdsi_quantiles

0.001   -5.211390
0.010   -4.439912
0.050   -3.498593
0.250   -1.750176
0.500   -0.566704
0.750    0.717594
0.950    2.962725
0.990    4.384193
0.999    5.790606
Name: pdsi, dtype: float64

In [40]:
for col in ["urban", "water_network", "water_shortage"]:
    print(f"\n--- {col} ---")
    print("Missing:", df[col].isna().sum())
    print("Unique values:", df[col].nunique())
    print("Values:", sorted(df[col].dropna().unique()))


--- urban ---
Missing: 0
Unique values: 534
Values: [np.float64(22.3), np.float64(26.74), np.float64(28.81), np.float64(33.12), np.float64(33.28), np.float64(34.27), np.float64(35.49), np.float64(36.18), np.float64(36.84), np.float64(37.32), np.float64(38.31), np.float64(38.43), np.float64(39.45), np.float64(39.94), np.float64(40.82), np.float64(40.83), np.float64(41.55), np.float64(42.0), np.float64(42.31), np.float64(42.47), np.float64(42.54), np.float64(42.82), np.float64(43.52), np.float64(43.98), np.float64(44.1), np.float64(44.19), np.float64(44.3), np.float64(44.37), np.float64(44.51), np.float64(45.95), np.float64(46.25), np.float64(46.97), np.float64(46.98), np.float64(47.04), np.float64(47.22), np.float64(47.56), np.float64(47.89), np.float64(47.93), np.float64(47.95), np.float64(48.36), np.float64(48.37), np.float64(48.53), np.float64(48.68), np.float64(48.9), np.float64(48.93), np.float64(48.98), np.float64(49.02), np.float64(49.06), np.float64(49.22), np.float64(49.29), n

In [41]:
context_cols = ["urban", "water_network", "water_shortage"]

context_summary = df[context_cols].agg(
    ["count", "min", "max", "mean", "std", "nunique"]
)

context_summary

,urban,water_network,water_shortage
count,133920.000000,133920.000000,133920.000000
min,22.300000,16.680000,0.000000
max,100.000000,98.810000,1.000000
mean,72.098190,72.286989,0.357296
std,16.630274,17.021309,0.240645
nunique,534.000000,528.000000,311.000000


In [42]:
context_consistency = (
    df.groupby(["micro_code", "year"])
      .agg(
          urban_min=("urban", "min"),
          urban_max=("urban", "max"),
          water_network_min=("water_network", "min"),
          water_network_max=("water_network", "max"),
          water_shortage_min=("water_shortage", "min"),
          water_shortage_max=("water_shortage", "max")
      )
      .reset_index()
)

context_consistency["urban_range"] = (
    context_consistency["urban_max"]
    - context_consistency["urban_min"]
)

context_consistency["water_network_range"] = (
    context_consistency["water_network_max"]
    - context_consistency["water_network_min"]
)

context_consistency["water_shortage_range"] = (
    context_consistency["water_shortage_max"]
    - context_consistency["water_shortage_min"]
)

context_consistency[
    [
        "urban_range",
        "water_network_range",
        "water_shortage_range"
    ]
].describe()

,urban_range,water_network_range,water_shortage_range
count,11160.0,11160.0,11160.0
mean,0.0,0.0,0.0
std,0.0,0.0,0.0
min,0.0,0.0,0.0
25%,0.0,0.0,0.0
50%,0.0,0.0,0.0
75%,0.0,0.0,0.0
max,0.0,0.0,0.0


In [43]:
context_yearly = (
    df.groupby(["micro_code", "year"])[
        ["urban", "water_network", "water_shortage"]
    ]
    .first()
    .reset_index()
)

context_changes = (
    context_yearly
    .sort_values(["micro_code", "year"])
    .groupby("micro_code")[
        ["urban", "water_network", "water_shortage"]
    ]
    .agg(["min", "max", "std"])
)

context_changes.describe()

urban                    water_network                     \
              min         max    std           min         max    std   
count  558.000000  558.000000  558.0    558.000000  558.000000  558.0   
mean    72.098190   72.098190    0.0     72.286989   72.286989    0.0   
std     16.645134   16.645134    0.0     17.036518   17.036518    0.0   
min     22.300000   22.300000    0.0     16.680000   16.680000    0.0   
25%     58.420000   58.420000    0.0     62.737500   62.737500    0.0   
50%     72.750000   72.750000    0.0     74.875000   74.875000    0.0   
75%     86.645000   86.645000    0.0     85.670000   85.670000    0.0   
max    100.000000  100.000000    0.0     98.810000   98.810000    0.0   

      water_shortage                     
                 min         max    std  
count     558.000000  558.000000  558.0  
mean        0.357296    0.357296    0.0  
std         0.240860    0.240860    0.0  
min         0.000000    0.000000    0.0  
25%         0.160100    0.160100    0.0  
50%         0.325850    0.325850    0.0  
75%         0.531550    0.531550    0.0  
max         1.000000    1.000000    0.0

In [44]:
climate_cols = ["tmax", "tmin", "pdsi"]

climate_summary = df[climate_cols].agg(
    ["count", "min", "max", "mean", "std"]
)

climate_summary

,tmax,tmin,pdsi
count,133920.000000,133920.000000,133680.000000
min,14.425291,4.615391,-7.634145
max,37.885544,25.765200,7.479940
mean,29.417284,18.980845,-0.459147
std,3.461089,3.644395,1.907180


In [45]:
climate_temporal = (
    df.groupby("micro_code")[["tmax", "tmin", "pdsi"]]
      .agg(["min", "max", "std"])
)

climate_temporal.describe()

tmax                                tmin                          \
              min         max         std         min         max         std   
count  558.000000  558.000000  558.000000  558.000000  558.000000  558.000000   
mean    25.365847   32.928781    1.854812   14.352022   22.387324    1.964582   
std      4.495794    2.166810    0.819995    5.215584    1.623798    1.010316   
min     14.425291   27.625614    0.583771    4.615391   18.239096    0.363889   
25%     22.547219   31.420270    1.319572    9.984578   21.200437    1.042763   
50%     26.444755   32.845266    1.654575   14.756877   22.512761    1.868012   
75%     29.114613   34.594800    2.045817   19.105981   23.685603    2.803011   
max     31.201759   37.885544    4.593399   22.707439   25.765200    4.206908   

             pdsi                          
              min         max         std  
count  557.000000  557.000000  557.000000  
mean    -3.939463    3.776401    1.667009  
std      0.997239    1.567602    0.472567  
min     -7.634145    0.270883    0.324528  
25%     -4.638983    2.416768    1.314552  
50%     -3.984386    3.793031    1.703036  
75%     -3.354835    5.074831    1.976655  
max     -0.885887    7.479940    2.782478

In [46]:
climate_monthly = (
    df.groupby("month")[["tmax", "tmin", "pdsi"]]
      .mean()
)

climate_monthly

,tmax,tmin,pdsi
month,,,
1,30.732663,20.891132,-0.346899
2,30.717141,20.920621,-0.383202
3,30.267476,20.600039,-0.414869
4,29.447483,19.718489,-0.414076
5,27.909896,17.878923,-0.416044
6,27.251661,16.516079,-0.425705
7,27.304951,15.964200,-0.431849
8,28.572144,16.692586,-0.518486
9,29.497723,18.226914,-0.569751


In [47]:
dengue_summary = df["dengue_cases"].agg(
    ["count", "min", "max", "mean", "std", "median"]
)

dengue_summary

count     127224.000000
min            0.000000
max        88821.000000
mean         101.358965
std          971.192888
median         4.000000
Name: dengue_cases, dtype: float64

In [48]:
dengue_zero_stats = {
    "observed_cases": df["dengue_cases"].notna().sum(),
    "zero_cases": (df["dengue_cases"] == 0).sum(),
    "positive_cases": (df["dengue_cases"] > 0).sum(),
    "zero_percentage": (
        100 * (df["dengue_cases"] == 0).sum()
        / df["dengue_cases"].notna().sum()
    )
}

dengue_zero_stats

{'observed_cases': np.int64(127224),
 'zero_cases': np.int64(34031),
 'positive_cases': np.int64(93193),
 'zero_percentage': np.float64(26.748883858391498)}

In [49]:
dengue_monthly = (
    df.groupby("month")["dengue_cases"]
      .agg(["count", "mean", "median"])
)

dengue_monthly

,count,mean,median
month,,,
1,10602,106.254197,6.0
2,10602,175.427372,9.0
3,10602,267.370307,17.5
4,10602,263.356819,18.0
5,10602,177.207980,14.0
6,10602,70.454820,6.0
7,10602,34.197038,3.0
8,10602,22.169119,2.0
9,10602,16.638936,1.0
